# Notebook 09 — Player Similarity Engine (FAISS)

Build a cosine similarity–based recommendation engine for FIFA players.

## 1. Imports ##

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR

import duckdb
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
import joblib
import faiss

## 2. Directories ##

In [3]:
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 3. Connection to DuckDB ##

In [4]:
con = duckdb.connect()

con.execute("PRAGMA threads=8")

PARQUET = PROCESSED_DATA_DIR / "players_clean"

## 4. Build Similarity Dataset ##

A sample of 300,000 players is usually sufficient for an interactive similarity engine while remaining memory-efficient.

In [5]:
similarity_df = con.sql(f"""
SELECT

player_id,
short_name,

TRY_CAST(age AS DOUBLE) AS age,
TRY_CAST(overall AS DOUBLE) AS overall,
TRY_CAST(potential AS DOUBLE) AS potential,
TRY_CAST(pace AS DOUBLE) AS pace,
TRY_CAST(shooting AS DOUBLE) AS shooting,
TRY_CAST(passing AS DOUBLE) AS passing,
TRY_CAST(dribbling AS DOUBLE) AS dribbling,
TRY_CAST(defending AS DOUBLE) AS defending,
TRY_CAST(physic AS DOUBLE) AS physic

FROM parquet_scan('{PARQUET}')

WHERE

TRY_CAST(overall AS DOUBLE) IS NOT NULL
AND TRY_CAST(potential AS DOUBLE) IS NOT NULL

USING SAMPLE 300000 ROWS
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 5. Clean Data ##

In [6]:
similarity_df = similarity_df.dropna()

similarity_df.shape

(266539, 11)

## 6. Build Feature Matrix ##

In [7]:
features = similarity_df.drop(
    columns=[
        "player_id",
        "short_name",
    ]
)

## 7. Standardize Features ##

In [8]:
scaler = StandardScaler()

X = scaler.fit_transform(features)

X = X.astype("float32")

## 8. Build FAISS Index ##

In [9]:
dimension = X.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(X)

print(index.ntotal)

266539


## 9. Similarity Search Function ##

In [10]:
def find_similar_players(player_name, k=10):
    
    matches = similarity_df[
        similarity_df["short_name"]
        .str.lower()
        == player_name.lower()
    ]

    if matches.empty:
        raise ValueError(f"{player_name} not found.")

    idx = matches.index[0]

    distances, indices = index.search(
        X[idx].reshape(1, -1),
        k + 1,
    )

    results = similarity_df.iloc[
        indices[0][1:]
    ].copy()

    results["distance"] = distances[0][1:]

    return results

## 10. Example Search ##

In [14]:
find_similar_players(
    "L. Messi",
    k=10,
)

,player_id,short_name,age,overall,potential,pace,shooting,passing,dribbling,defending,physic,distance
185074,234164,Fábio China,25.0,68.0,71.0,68.0,39.0,56.0,65.0,66.0,63.0,0.033500
1779,234164,Fábio China,24.0,69.0,72.0,68.0,39.0,56.0,65.0,67.0,63.0,0.083067
179881,234164,Fábio China,26.0,69.0,69.0,68.0,39.0,57.0,65.0,67.0,64.0,0.163441
286937,234164,Fábio China,26.0,69.0,69.0,68.0,39.0,57.0,65.0,67.0,64.0,0.163441
50415,234164,Fábio China,24.0,70.0,73.0,71.0,39.0,57.0,65.0,68.0,63.0,0.252025
165438,234164,Fábio China,24.0,70.0,73.0,71.0,39.0,57.0,65.0,68.0,63.0,0.252025
183362,235548,M. Cabit,26.0,67.0,70.0,70.0,38.0,59.0,62.0,65.0,64.0,0.341593
258256,235548,M. Cabit,26.0,67.0,70.0,70.0,38.0,59.0,62.0,65.0,64.0,0.341593
114703,234164,Fábio China,27.0,69.0,69.0,70.0,39.0,57.0,64.0,67.0,62.0,0.349197
177085,234164,Fábio China,27.0,69.0,69.0,70.0,39.0,57.0,64.0,67.0,62.0,0.349197


## 11. Save FAISS Index ##

In [15]:
faiss.write_index(
    index,
    str(MODELS_DIR / "player_similarity.index"),
)

## 12. Save Scaler ##

In [16]:
joblib.dump(
    scaler,
    MODELS_DIR / "similarity_scaler.pkl",
)

['/Users/subhankarbiswas/fifa-player-analytics/models/similarity_scaler.pkl']

## 13. Save Metadata ##

In [17]:
metadata = similarity_df[
    [
        "player_id",
        "short_name",
    ]
]

metadata.to_parquet(
    PROCESSED_DATA_DIR / "similarity_metadata.parquet",
    index=False,
)

## 14. Save Feature Matrix ##

In [18]:
np.save(
    MODELS_DIR / "similarity_vectors.npy",
    X,
)

## Suggested analyses

- Compare recommended players with the selected player.
- Evaluate how engineered features influence recommendations.
- Experiment with different feature subsets.

### Next

Proceed to **Notebook 10 — Final Report & Project Summary**.